In [5]:
import pandas as pd

results = pd.read_csv("../data/raw/combined_match_download_results.csv")

total = len(results)
matches = results['match_found'].sum()
downloads = results['download_success'].sum()

print(f"Progress: {total:,} songs processed")
print(f"Matches: {matches:,} ({matches/total*100:.1f}%)")
print(f"Downloads: {downloads:,} ({downloads/matches*100:.1f}% of matches)")
print(f"\nAt this pace, completion in ~{(57000-total)*2.5/3600:.1f} hours")

Progress: 47,344 songs processed
Matches: 47,344 (100.0%)
Downloads: 45,790 (96.7% of matches)

At this pace, completion in ~6.7 hours


In [4]:
from pathlib import Path

audio_dir = Path("../data/raw/youtube_audio/")
mp3_files = list(audio_dir.glob("*.mp3"))

if len(mp3_files) > 0:
    total_size_mb = sum(f.stat().st_size for f in mp3_files) / (1024 * 1024)
    avg_size_mb = total_size_mb / len(mp3_files)
    
    print(f"Files: {len(mp3_files)}")
    print(f"Total size: {total_size_mb:.2f} MB ({total_size_mb/1024:.2f} GB)")
    print(f"Average per file: {avg_size_mb:.2f} MB")
    
    # Project for 57k
    projected_gb = (avg_size_mb * 57000) / 1024
    print(f"\nProjected for 57,000 songs: {projected_gb:.1f} GB")

Files: 44271
Total size: 157025.67 MB (153.35 GB)
Average per file: 3.55 MB

Projected for 57,000 songs: 197.4 GB


In [13]:
import pandas as pd
import json
import subprocess

# Load one of your matched videos
results = pd.read_csv("../data/raw/combined_match_download_results.csv")
sample = results[results['match_found'] == True].iloc[0]

video_id = sample['youtube_video_id']

# Get full metadata
cmd = ['yt-dlp', f'https://www.youtube.com/watch?v={video_id}', '--dump-json', '--skip-download']
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    data = json.loads(result.stdout)
    print("Available metadata fields:")
    for key in sorted(data.keys()):
        print(f"  {key}: {type(data[key]).__name__}")

Available metadata fields:
  _filename: str
  _format_sort_fields: list
  _has_drm: NoneType
  _type: str
  _version: dict
  abr: float
  acodec: str
  age_limit: int
  aspect_ratio: float
  asr: int
  audio_channels: int
  automatic_captions: dict
  availability: str
  average_rating: NoneType
  categories: list
  channel: str
  channel_follower_count: int
  channel_id: str
  channel_is_verified: bool
  channel_url: str
  chapters: NoneType
  comment_count: int
  creators: NoneType
  description: str
  display_id: str
  duration: int
  duration_string: str
  dynamic_range: str
  epoch: int
  ext: str
  extractor: str
  extractor_key: str
  filename: str
  filesize_approx: int
  format: str
  format_id: str
  format_note: str
  formats: list
  fps: int
  fulltitle: str
  heatmap: list
  height: int
  id: str
  is_live: bool
  language: NoneType
  like_count: int
  live_status: str
  media_type: str
  original_url: str
  playable_in_embed: bool
  playlist: NoneType
  playlist_index: Non

In [17]:
import pandas as pd
import json
import subprocess

# Load your current results
results = pd.read_csv("../data/raw/combined_match_download_results.csv")

# Get a matched video
matched = results[results['match_found'] == True].iloc[0]

print("=" * 70)
print("SONG INFORMATION")
print("=" * 70)
print(f"Track Name: {matched['track_name']}")
print(f"Artist: {matched['artists']}")
print(f"Album: {matched.get('album_name', 'N/A')}")
print(f"Spotify Popularity: {matched.get('popularity', 'N/A')}")
print(f"\nYouTube Video ID: {matched['youtube_video_id']}")
print(f"YouTube URL: {matched['youtube_url']}")
print(f"YouTube Title: {matched['youtube_title']}")
print(f"YouTube Channel: {matched['youtube_channel']}")
print(f"Match Confidence: {matched.get('match_confidence', 'N/A')}")

# Get full metadata
video_id = matched['youtube_video_id']
cmd = ['yt-dlp', f'https://www.youtube.com/watch?v={video_id}', '--dump-json', '--skip-download', '--no-warnings', '--quiet']
result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)

if result.returncode == 0:
    data = json.loads(result.stdout)
    
    print("\n" + "=" * 70)
    print("METADATA FIELDS COLLECTED (10 CORE FIELDS)")
    print("=" * 70)
    
    # 1-4: Engagement metrics
    print("\nENGAGEMENT METRICS:")
    print(f"  1. view_count: {data.get('view_count', 0):,}")
    print(f"  2. like_count: {data.get('like_count', 0):,}")
    print(f"  3. comment_count: {data.get('comment_count', 0):,}")
    print(f"  4. channel_follower_count: {data.get('channel_follower_count', 0):,}")
    
    # 5-6: Content info
    print("\nCONTENT INFORMATION:")
    print(f"  5. duration: {data.get('duration', 0)} seconds ({data.get('duration', 0)/60:.1f} minutes)")
    print(f"  6. upload_date: {data.get('upload_date', 'Unknown')}")
    
    # 7: Heatmap
    heatmap = data.get('heatmap', [])
    print("\nENGAGEMENT PATTERN:")
    print(f"  7. heatmap: {len(heatmap)} data points")
    if heatmap:
        avg_engagement = sum(h.get('value', 0) for h in heatmap) / len(heatmap) if heatmap else 0
        max_engagement = max(h.get('value', 0) for h in heatmap) if heatmap else 0
        print(f"     Average engagement: {avg_engagement:.2f}")
        print(f"     Peak engagement: {max_engagement:.2f}")
        print(f"     Sample data: {heatmap[:3]}")
    
    # 8-10: Classification
    print("\nCLASSIFICATION:")
    print(f"  8. channel_is_verified: {data.get('channel_is_verified', False)}")
    
    tags = data.get('tags', [])
    print(f"  9. tags: {len(tags)} tags total")
    if tags:
        print(f"     Sample tags: {tags[:10]}")
    
    categories = data.get('categories', [])
    print(f" 10. categories: {categories}")
    
    # Additional useful fields
    print("\n" + "=" * 70)
    print("ADDITIONAL AVAILABLE FIELDS")
    print("=" * 70)
    
    print("\nCHANNEL INFORMATION:")
    print(f"  channel: {data.get('channel', 'N/A')}")
    print(f"  channel_id: {data.get('channel_id', 'N/A')}")
    print(f"  channel_url: {data.get('channel_url', 'N/A')}")
    
    print("\nVIDEO DETAILS:")
    print(f"  title: {data.get('title', 'N/A')}")
    print(f"  description: {data.get('description', 'N/A')[:200]}...")
    print(f"  thumbnail: {data.get('thumbnail', 'N/A')}")
    
    print("\nTECHNICAL SPECS:")
    print(f"  width: {data.get('width', 'N/A')}")
    print(f"  height: {data.get('height', 'N/A')}")
    print(f"  fps: {data.get('fps', 'N/A')}")
    print(f"  duration_string: {data.get('duration_string', 'N/A')}")
    
    print("\nCONTENT FLAGS:")
    print(f"  age_limit: {data.get('age_limit', 0)}")
    print(f"  is_live: {data.get('is_live', False)}")
    print(f"  was_live: {data.get('was_live', False)}")
    print(f"  availability: {data.get('availability', 'N/A')}")
    print(f"  playable_in_embed: {data.get('playable_in_embed', 'N/A')}")
    
    # Derived metrics for analysis
    print("\n" + "=" * 70)
    print("DERIVED METRICS (FOR VIRALITY ANALYSIS)")
    print("=" * 70)
    
    view_count = data.get('view_count', 0)
    like_count = data.get('like_count', 0)
    comment_count = data.get('comment_count', 0)
    channel_followers = data.get('channel_follower_count', 0)
    
    if view_count > 0:
        print(f"\nEngagement Rate (likes/views): {(like_count/view_count)*100:.3f}%")
        print(f"Comment Rate (comments/views): {(comment_count/view_count)*100:.4f}%")
        
        if channel_followers > 0:
            print(f"Reach Efficiency (views/subscribers): {view_count/channel_followers:.2f}x")
    
    # Upload recency
    upload_date_str = data.get('upload_date', '')
    if upload_date_str:
        from datetime import datetime
        try:
            upload_dt = datetime.strptime(upload_date_str, '%Y%m%d')
            days_old = (datetime.now() - upload_dt).days
            years_old = days_old / 365.25
            print(f"\nVideo Age: {days_old} days ({years_old:.1f} years)")
            if days_old > 0:
                print(f"Views per Day: {view_count/days_old:,.0f}")
                print(f"Likes per Day: {like_count/days_old:,.0f}")
        except:
            pass
    
    print("\n" + "=" * 70)
    print("DATA COLLECTION COMPLETE")
    print("=" * 70)
    
else:
    print("\nFailed to retrieve metadata")
    print(f"Error: {result.stderr}")

SONG INFORMATION
Track Name: Say Something
Artist: A Great Big World;Christina Aguilera
Album: Is There Anybody Out There?
Spotify Popularity: 74.0

YouTube Video ID: -2U0Ivkn2Ds
YouTube URL: https://www.youtube.com/watch?v=-2U0Ivkn2Ds
YouTube Title: A Great Big World, Christina Aguilera - Say Something (Official Video)
YouTube Channel: A Great Big World
Match Confidence: 1.0

METADATA FIELDS COLLECTED (10 CORE FIELDS)

ENGAGEMENT METRICS:
  1. view_count: 766,742,700
  2. like_count: 5,008,190
  3. comment_count: 139,000
  4. channel_follower_count: 1,300,000

CONTENT INFORMATION:
  5. duration: 231 seconds (3.9 minutes)
  6. upload_date: 20131120

ENGAGEMENT PATTERN:
  7. heatmap: 100 data points
     Average engagement: 0.87
     Peak engagement: 1.00
     Sample data: [{'start_time': 0.0, 'end_time': 2.32, 'value': 0.9878978986012337}, {'start_time': 2.32, 'end_time': 4.64, 'value': 0.7414261525068638}, {'start_time': 4.64, 'end_time': 6.96, 'value': 0.8222913932145499}]

CLASSIFIC

In [7]:
import pandas as pd
import numpy as np

# Load CSV
results = pd.read_csv("../data/raw/combined_match_download_results.csv")

print("=" * 70)
print("BATCH FAILURE ANALYSIS")
print("=" * 70)

# Analyze in chunks of 100 (batch size)
batch_size = 100
total_songs = len(results)
num_batches = int(np.ceil(total_songs / batch_size))

print(f"\nTotal songs: {total_songs:,}")
print(f"Total batches: {num_batches}")
print(f"Batch size: {batch_size}\n")

# Track problematic batches
failed_batches = []
low_success_batches = []

for batch_num in range(num_batches):
    start_idx = batch_num * batch_size
    end_idx = min(start_idx + batch_size, total_songs)
    batch = results.iloc[start_idx:end_idx]
    
    matches = batch['match_found'].sum()
    downloads = batch['download_success'].sum()
    batch_len = len(batch)
    
    match_rate = (matches / batch_len) * 100 if batch_len > 0 else 0
    download_rate = (downloads / matches) * 100 if matches > 0 else 0
    
    # Flag problematic batches
    if matches == 0 and downloads == 0:
        failed_batches.append({
            'batch_num': batch_num + 1,
            'songs': f"{start_idx + 1}-{end_idx}",
            'matches': matches,
            'downloads': downloads,
            'match_rate': match_rate
        })
    elif match_rate < 50 or (matches > 0 and download_rate < 50):
        low_success_batches.append({
            'batch_num': batch_num + 1,
            'songs': f"{start_idx + 1}-{end_idx}",
            'matches': matches,
            'downloads': downloads,
            'match_rate': match_rate,
            'download_rate': download_rate
        })

# Report failed batches (0/0)
print("=" * 70)
print("COMPLETELY FAILED BATCHES (0 matches, 0 downloads)")
print("=" * 70)

if failed_batches:
    print(f"\nFound {len(failed_batches)} completely failed batches:\n")
    for batch in failed_batches:
        print(f"  Batch {batch['batch_num']} (songs {batch['songs']})")
        print(f"    Matches: {batch['matches']}/{batch_size}")
        print(f"    Downloads: {batch['downloads']}/{batch_size}")
        print()
else:
    print("\n✓ No completely failed batches found!")

# Report low success batches
print("=" * 70)
print("LOW SUCCESS BATCHES (<50% match or download rate)")
print("=" * 70)

if low_success_batches:
    print(f"\nFound {len(low_success_batches)} low success batches:\n")
    for batch in low_success_batches:
        print(f"  Batch {batch['batch_num']} (songs {batch['songs']})")
        print(f"    Matches: {batch['matches']}/{batch_size} ({batch['match_rate']:.1f}%)")
        print(f"    Downloads: {batch['downloads']}/{batch['matches'] if batch['matches'] > 0 else batch_size} ({batch['download_rate']:.1f}%)")
        print()
else:
    print("\n✓ No low success batches found!")

# Overall statistics
print("=" * 70)
print("OVERALL STATISTICS")
print("=" * 70)

total_matches = results['match_found'].sum()
total_downloads = results['download_success'].sum()

print(f"\nTotal processed: {total_songs:,}")
print(f"Total matches: {total_matches:,} ({(total_matches/total_songs)*100:.1f}%)")
print(f"Total downloads: {total_downloads:,} ({(total_downloads/total_matches)*100:.1f}% of matches)")

# Success rate distribution
print(f"\nBatch success distribution:")
batch_match_rates = []
for batch_num in range(num_batches):
    start_idx = batch_num * batch_size
    end_idx = min(start_idx + batch_size, total_songs)
    batch = results.iloc[start_idx:end_idx]
    match_rate = (batch['match_found'].sum() / len(batch)) * 100
    batch_match_rates.append(match_rate)

print(f"  Average batch match rate: {np.mean(batch_match_rates):.1f}%")
print(f"  Median batch match rate: {np.median(batch_match_rates):.1f}%")
print(f"  Min batch match rate: {np.min(batch_match_rates):.1f}%")
print(f"  Max batch match rate: {np.max(batch_match_rates):.1f}%")

# Distribution by ranges
print(f"\nBatch match rate distribution:")
print(f"  100%: {sum(1 for r in batch_match_rates if r == 100)} batches")
print(f"  90-99%: {sum(1 for r in batch_match_rates if 90 <= r < 100)} batches")
print(f"  80-89%: {sum(1 for r in batch_match_rates if 80 <= r < 90)} batches")
print(f"  50-79%: {sum(1 for r in batch_match_rates if 50 <= r < 80)} batches")
print(f"  1-49%: {sum(1 for r in batch_match_rates if 1 <= r < 50)} batches")
print(f"  0%: {sum(1 for r in batch_match_rates if r == 0)} batches")

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)

BATCH FAILURE ANALYSIS

Total songs: 51,579
Total batches: 516
Batch size: 100

COMPLETELY FAILED BATCHES (0 matches, 0 downloads)

Found 16 completely failed batches:

  Batch 463 (songs 46201-46300)
    Matches: 0/100
    Downloads: 0/100

  Batch 464 (songs 46301-46400)
    Matches: 0/100
    Downloads: 0/100

  Batch 465 (songs 46401-46500)
    Matches: 0/100
    Downloads: 0/100

  Batch 466 (songs 46501-46600)
    Matches: 0/100
    Downloads: 0/100

  Batch 467 (songs 46601-46700)
    Matches: 0/100
    Downloads: 0/100

  Batch 468 (songs 46701-46800)
    Matches: 0/100
    Downloads: 0/100

  Batch 469 (songs 46801-46900)
    Matches: 0/100
    Downloads: 0/100

  Batch 470 (songs 46901-47000)
    Matches: 0/100
    Downloads: 0/100

  Batch 471 (songs 47001-47100)
    Matches: 0/100
    Downloads: 0/100

  Batch 472 (songs 47101-47200)
    Matches: 0/100
    Downloads: 0/100

  Batch 473 (songs 47201-47300)
    Matches: 0/100
    Downloads: 0/100

  Batch 474 (songs 47301-474

In [1]:
import pandas as pd

# Load spotify dataset
spotify_df = pd.read_csv("../data/raw/spotify_tracks.csv")

# Check what START_INDEX is set to
print(f"START_INDEX: {START_INDEX}")
print(f"END_INDEX: {END_INDEX}")

# Look at first few songs being processed
print("\nFirst 5 songs in processing range:")
for i in range(START_INDEX, START_INDEX + 5):
    song = spotify_df.iloc[i]
    print(f"  {i}: {song['track_name']} - {song['artists']}")

NameError: name 'START_INDEX' is not defined

In [1]:
import requests
import subprocess
import json
from pathlib import Path

print("="*70)
print("YOUTUBE BLOCK TEST")
print("="*70)

# 1. Check IP
print("\n1. Checking IP address...")
try:
    ip = requests.get('https://api.ipify.org', timeout=10).text
    print(f"   Current IP: {ip}")
except:
    print("   ✗ Couldn't check IP")

# 2. Test YouTube search
print("\n2. Testing YouTube search...")
cmd = ['yt-dlp', 'ytsearch1:test', '--dump-json', '--skip-download']
try:
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
    
    if 'sign in to confirm' in result.stderr.lower():
        print("   ✗ BLOCKED: 'Sign in to confirm' error")
        blocked = True
    elif 'bot' in result.stderr.lower():
        print("   ✗ BLOCKED: Bot detection")
        blocked = True
    elif result.stdout and len(result.stdout) > 50:
        print("   ✓ UNBLOCKED: Search successful")
        blocked = False
    else:
        print("   ⚠️ UNCLEAR")
        blocked = None
except Exception as e:
    print(f"   ✗ Error: {e}")
    blocked = None

# 3. Summary
print("\n" + "="*70)
if blocked == False:
    print("RESULT: ✓ YouTube is UNBLOCKED!")
    print("You can now collect metadata!")
elif blocked == True:
    print("RESULT: ✗ Still BLOCKED")
    print("Wait longer, restart router again, or use VPN")
else:
    print("RESULT: ⚠️ Status unclear - try manual test")
print("="*70)

YOUTUBE BLOCK TEST

1. Checking IP address...
   Current IP: 121.6.238.191

2. Testing YouTube search...
   ✗ BLOCKED: 'Sign in to confirm' error

RESULT: ✗ Still BLOCKED
Wait longer, restart router again, or use VPN
